# Zernike响应矩阵校准 - 硬件调试notebook

本notebook用于实际硬件调试，执行Zernike响应矩阵校准。

## 硬件连接
- **SLM**: Santec SLM-200 (通过ZernikeSLM封装)
- **WFS**: Thorlabs WFS

## 校准原理

使用正负扰动测量消除系统偏置:

$R = \frac{R^+ - R^-}{2\cdot\delta}$

其中 $R^+$, $R^-$ 是正负扰动下的WFS响应，$\delta$ 是扰动幅度(波长单位)。

In [ ]:
# %% [markdown]
# =============================================================================
# 🎯 关键配置参数 (请先修改此处)
# =============================================================================

# -------------------------------- 校准参数 --------------------------------
N_MAX = 10                      # Zernike最大阶数 (建议: 4-15)
MAGNITUDE = 0.5                 # 扰动幅度 (波长单位, 建议: 0.3-1.0)
N_CYCLES = 3                    # 正负交替循环次数 (建议: 1-5)
N_AVERAGES = 20                 # 每次WFS读取取平均次数 (建议: 10-50)
WAIT_TIME = 0.1                 # 施加相位后等待时间 (秒, 建议: 0.05-0.5)
EXCLUDED_PISTON = True          # 是否排除piston (Z1)
EXCLUDED_TIP_TILT = True        # 是否排除tip/tilt (Z2, Z3)

# -------------------------------- 硬件参数 --------------------------------
# SLM
SLM_NUMBER = 1                  # SLM设备号
SLM_WAVELENGTH = 1064           # 工作波长 (nm)

# SLM过冲控制
SLM_MAX_PHASE_JUMP = 0.3        # 最大相位跳变 (λ), 超过则分步发送
SLM_STEP_SIZE = 0.1             # 分步发送步长 (λ)
SLM_STEP_DELAY = 0.05           # 步间延迟 (秒)

# WFS
WFS_RESOLUTION = 768            # 分辨率: 320, 512, 768, 1024, 1280
WFS_EXPOSURE_TIME = 10.0        # 曝光时间 (ms), 0=自动

# WFS稳定判断
WFS_STABLE_RMS_THRESHOLD = 0.01 # RMS稳定阈值 (λ)
WFS_STABLE_FRAMES = 3           # 连续多少帧RMS变化小于阈值认为稳定
WFS_STABLE_WARMUP = 10          # 稳定判断前预热帧数

# -------------------------------- 输出路径 --------------------------------
OUTPUT_DIR = "data/zernike_response_matrix"  # 结果保存目录

# =============================================================================
# 提示: 修改上述参数后，从下一个cell开始执行
# =============================================================================

In [ ]:
# %% [markdown]
# ## 1. 导入核心模块 + 定义辅助函数

import sys
sys.path.insert(0, 'src')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time
from datetime import datetime

# Zernike响应矩阵模块
from ao_shaping.optimizer.wf.zernike_response_matrix import (
    calibrate_zernike_response_matrix,
    measure_zernike_mode_response,
    ZernikeResponseMatrixResult,
    set_slm_flat,
    save_zernike_response_matrix,
)

# 工具函数
from ao_shaping.utils.matrix_utils import calc_n_zernike_terms

# WFS分辨率枚举
from ao_shaping.drivers.wfs.thorlab_wfs import MlaRes

# 解析WFS分辨率
WFS_RESOLUTION_MAP = {320: MlaRes.Res320, 512: MlaRes.Res512, 768: MlaRes.Res768, 1024: MlaRes.Res1024, 1280: MlaRes.Res1280}
wfs_res_enum = WFS_RESOLUTION_MAP.get(WFS_RESOLUTION, MlaRes.Res768)

# 创建输出目录
OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 计算项数
n_remove = (1 if EXCLUDED_PISTON else 0) + (2 if EXCLUDED_TIP_TILT else 0)
n_slm_terms = calc_n_zernike_terms(N_MAX) - n_remove

print("="*60)
print("✓ 模块导入成功")
print("="*60)
print(f"【校准参数】n_max={N_MAX}, mag={MAGNITUDE}λ, cycles={N_CYCLES}, avg={N_AVERAGES}")
print(f"【过冲控制】max_jump={SLM_MAX_PHASE_JUMP}λ, step={SLM_STEP_SIZE}λ, delay={SLM_STEP_DELAY}s")
print(f"【WFS稳定】threshold={WFS_STABLE_RMS_THRESHOLD}λ, warmup={WFS_STABLE_WARMUP}, stable_frames={WFS_STABLE_FRAMES}")
print(f"【SLM项数】{n_slm_terms}")
print("="*60)

# =============================================================================
# 辅助函数: SLM过冲控制
# =============================================================================
def send_phase_overshoot_control(zslm, target_coeffs, max_jump=0.3, step_size=0.1, step_delay=0.05):
    """分步发送相位，防止SLM过冲"""
    current = zslm.get_current_zernike_coeffs()
    current_arr = np.array(list(current.values())) if isinstance(current, dict) else (current if current is not None else np.zeros_like(target_coeffs))
    if current_arr.shape != target_coeffs.shape:
        current_arr = np.zeros_like(target_coeffs)
    
    max_change = np.max(np.abs(target_coeffs - current_arr))
    
    if max_change <= max_jump:
        return zslm.send_zernike(target_coeffs)
    else:
        n_steps = int(np.ceil(max_change / step_size))
        print(f"  ⚠ 过冲控制: {max_change:.3f}λ > {max_jump}λ, 分{n_steps}步")
        for i in range(n_steps):
            fraction = (i + 1) / n_steps
            interp = current_arr + (target_coeffs - current_arr) * fraction
            zslm.send_zernike(interp)
            time.sleep(step_delay)
        return zslm.get_current_phase()

# =============================================================================
# 辅助函数: WFS RMS稳定检测
# =============================================================================
def wait_wfs_stable(wfs, n_warmup=10, threshold=0.01, stable_frames=3, zernike_order=10):
    """等待WFS RMS稳定"""
    rms_history = []
    
    # 预热
    print(f"  预热WFS ({n_warmup}帧)...")
    for _ in range(n_warmup):
        wfs.take_image()
        c = wfs.get_zernike(zernike_order=zernike_order)
        rms_history.append(np.sqrt(np.mean(c[3:]**2)))
        time.sleep(0.05)
    print(f"  预热完成 RMS={rms_history[-1]:.4f}λ")
    
    # 稳定检测
    n_stable = 0
    n_total = n_warmup
    while n_stable < stable_frames:
        wfs.take_image()
        c = wfs.get_zernike(zernike_order=zernike_order)
        rms = np.sqrt(np.mean(c[3:]**2))
        rms_history.append(rms)
        n_total += 1
        if len(rms_history) >= 2 and abs(rms - rms_history[-2]) < threshold:
            n_stable += 1
        else:
            n_stable = 0
        time.sleep(0.05)
        if n_total > n_warmup + 50:
            print(f"  ⚠ 达到最大帧数, RMS={rms:.4f}λ")
            break
    
    print(f"  ✓ WFS稳定: RMS={rms:.4f}λ, 共{n_total}帧")
    return {'rms_history': rms_history, 'stable_rms': rms, 'n_total': n_total}

print("✓ 辅助函数已定义")

In [ ]:
# %% [markdown]
# ## 2. 初始化硬件设备

from ao_shaping.drivers.slm.zernike_slm import ZernikeSLM
from ao_shaping.drivers.wfs.thorlab_wfs import WFSManager

print("初始化设备...")
print(f"  SLM: #{SLM_NUMBER}, λ={SLM_WAVELENGTH}nm, n_max={N_MAX}")
print(f"  WFS: {WFS_RESOLUTION}px, 曝光={WFS_EXPOSURE_TIME}ms")

zslm = ZernikeSLM(slm_number=SLM_NUMBER, wavelength=SLM_WAVELENGTH, n_max=N_MAX)
wfs = WFSManager(mla_index=wfs_res_enum, exp_time=WFS_EXPOSURE_TIME)

try:
    zslm.open()
    print("✓ SLM已打开")
    wfs.initialize()
    print("✓ WFS已初始化")
    print(f"  WFS: {wfs.serial_num}, {wfs.mla_index.name}, {wfs.num_spots_x}x{wfs.num_spots_y}")
    DEVICE_READY = True
except Exception as e:
    print(f"✗ 设备初始化失败: {e}")
    DEVICE_READY = False

In [ ]:
# %% [markdown]
# ## 3. SLM过冲测试 + WFS稳定预热

if not DEVICE_READY:
    raise RuntimeError("设备未就绪")

# --- 3.1 SLM过冲测试 ---
print("="*60)
print("SLM过冲控制测试")
print("="*60)

test_coeffs = np.zeros(calc_n_zernike_terms(N_MAX), dtype=np.float64)
test_coeffs[4], test_coeffs[5], test_coeffs[6] = 0.3, 0.2, 0.15
print(f"测试相位: Z4={test_coeffs[4]:.2f}λ, Z5={test_coeffs[5]:.2f}λ, Z6={test_coeffs[6]:.2f}λ")

set_slm_flat(zslm._slm)
time.sleep(0.2)

_ = send_phase_overshoot_control(zslm, test_coeffs, SLM_MAX_PHASE_JUMP, SLM_STEP_SIZE, SLM_STEP_DELAY)
_ = send_phase_overshoot_control(zslm, np.zeros_like(test_coeffs), SLM_MAX_PHASE_JUMP, SLM_STEP_SIZE, SLM_STEP_DELAY)

large = np.zeros_like(test_coeffs)
large[7] = 0.8
print(f"大幅跳变: Z7={large[7]:.2f}λ")
_ = send_phase_overshoot_control(zslm, large, SLM_MAX_PHASE_JUMP, SLM_STEP_SIZE, SLM_STEP_DELAY)
set_slm_flat(zslm._slm)
print("✓ SLM过冲测试完成")

# --- 3.2 WFS稳定预热 ---
print("\n" + "="*60)
print("WFS稳定预热")
print("="*60)
info = wait_wfs_stable(wfs, WFS_STABLE_WARMUP, WFS_STABLE_RMS_THRESHOLD, WFS_STABLE_FRAMES, N_MAX)

plt.figure(figsize=(10, 4))
plt.plot(info['rms_history'], 'o-', markersize=3)
plt.axhline(info['stable_rms'], color='g', linestyle='--')
plt.xlabel('Frame')
plt.ylabel('RMS (λ)')
plt.title('WFS预热 RMS变化')
plt.grid(True, alpha=0.3)
plt.show()
print(f"\n✓ 预热完成, RMS={info['stable_rms']:.4f}λ")

In [ ]:
# %% [markdown]
# ## 4. 快速单模式测试

TEST_MODE = 1
TEST_MAG = 0.5

print("="*60)
print(f"单模式测试: mode={TEST_MODE}, mag={TEST_MAG}λ")
print("="*60)

n_full = wfs.calc_n_zernike_terms(N_MAX)
coeffs = np.zeros(n_full, dtype=np.float64)
coeffs[TEST_MODE + 3] = TEST_MAG

wfs.take_image()
time.sleep(0.1)

print("发送相位 (过冲控制)...")
_ = send_phase_overshoot_control(zslm, coeffs, SLM_MAX_PHASE_JUMP, SLM_STEP_SIZE, SLM_STEP_DELAY)

mean_r, var_r, _, _ = measure_zernike_mode_response(
    zslm, wfs, coeffs, TEST_MAG, n_averages=10, n_cycles=1, wait_time=WAIT_TIME,
    excluded_piston=True, excluded_tip_tilt=True, zernike_order=N_MAX, mode_index=TEST_MODE
)

_ = send_phase_overshoot_control(zslm, np.zeros(n_full), SLM_MAX_PHASE_JUMP, SLM_STEP_SIZE, SLM_STEP_DELAY)

print(f"响应RMS: {np.sqrt(np.mean(mean_r**2)):.6f}, 平均方差: {np.mean(var_r):.6f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(range(len(mean_r)), mean_r, color='steelblue')
ax[0].set_title(f'响应 (Z{TEST_MODE+4})')
ax[0].set_xlabel('WFS Mode')
ax[1].bar(range(len(var_r)), var_r, color='coral')
ax[1].set_title('方差')
ax[1].set_xlabel('WFS Mode')
plt.tight_layout()
plt.show()
print("✓ 单模式测试完成")

In [ ]:
# %% [markdown]
# ## 5. 执行完整响应矩阵校准

print("="*60)
print("开始校准")
print("="*60)
print(f"参数: n_max={N_MAX}, mag={MAGNITUDE}λ, cycles={N_CYCLES}, avg={N_AVERAGES}")
print(f"预计: ~{n_slm_terms * N_CYCLES * N_AVERAGES * 0.5 / 60:.1f} 分钟")

# 校准前确认WFS稳定
wfs.take_image()
rms_now = np.sqrt(np.mean(wfs.get_zernike(N_MAX)[3:]**2))
print(f"当前RMS: {rms_now:.4f}λ")

set_slm_flat(zslm._slm)
time.sleep(0.5)

start = time.time()
result = calibrate_zernike_response_matrix(
    zslm, wfs, N_MAX, MAGNITUDE, N_CYCLES, N_AVERAGES, WAIT_TIME,
    EXCLUDED_PISTON, EXCLUDED_TIP_TILT, compute_inverses=True, verbose=True
)

print("="*60)
print(f"校准完成! 耗时: {(time.time()-start)/60:.1f}分钟")
print("="*60)
print(f"矩阵: {result.matrix.shape}, 方差: {result.mean_variance:.6f}, 条件数: {result.condition_number:.2e}")

In [ ]:
# %% [markdown]
# ## 6. 保存校准结果

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
h5_path = OUTPUT_DIR / f"zernike_matrix_{ts}.h5"
save_zernike_response_matrix(result, str(h5_path), True)

json_path = OUTPUT_DIR / f"zernike_matrix_{ts}.json"
import json
with open(json_path, 'w') as f:
    json.dump({k: v for k, v in result.to_dict().items() if k not in ['pinv_matrix', 'lstsq_matrix', 'deviation_response_matrix', 'subaperture_mask']}, f, indent=2)
print(f"✓ 已保存: {h5_path}")

In [ ]:
# %% [markdown]
# ## 7. 可视化分析

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

im1 = axes[0,0].imshow(result.matrix, aspect='auto', cmap='RdBu_r', vmin=-np.max(np.abs(result.matrix))*0.5, vmax=np.max(np.abs(result.matrix))*0.5)
axes[0,0].set_title('响应矩阵')
axes[0,0].set_xlabel('SLM Mode')
axes[0,0].set_ylabel('WFS Mode')
plt.colorbar(im1, ax=axes[0,0])

im2 = axes[0,1].imshow(result.variance_matrix, aspect='auto', cmap='YlOrRd')
axes[0,1].set_title(f'方差矩阵 (mean={result.mean_variance:.6f})')
plt.colorbar(im2, ax=axes[0,1])

axes[1,0].bar(range(len(np.mean(result.variance_matrix, axis=0))), np.mean(result.variance_matrix, axis=0))
axes[1,0].set_title('各模式稳定性')
axes[1,0].set_xlabel('SLM Mode')
axes[1,0].grid(True, alpha=0.3)

U, s, Vt = np.linalg.svd(result.matrix)
axes[1,1].plot(s, 'o-', markersize=4, color='darkorange')
axes[1,1].set_yscale('log')
axes[1,1].set_title(f'SVD (cond={result.condition_number:.2e})')
axes[1,1].set_xlabel('Singular Value')
axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# %% [markdown]
# ## 8. 验证逆矩阵

if result.pinv_matrix is not None:
    wfs.take_image()
    wf = wfs.get_zernike(N_MAX)
    vec = wf[3:3+n_slm_terms]
    corr = result.pinv_matrix @ vec
    resid = result.matrix @ corr - vec
    rms_b, rms_a = np.sqrt(np.mean(vec**2)), np.sqrt(np.mean(resid**2))
    print(f"校正前: {rms_b:.4f}λ, 校正后: {rms_a:.4f}λ, 改善: {(rms_b-rms_a)/rms_b*100:.1f}%")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].bar(range(len(vec)), vec, color='coral')
    axes[0].set_title('原始波前')
    axes[1].bar(range(len(corr)), corr, color='steelblue')
    axes[1].set_title('校正指令')
    axes[2].bar(range(len(resid)), resid, color='forestgreen')
axes[2].set_title(f'残余 ({rms_a:.4f}λ)')
    plt.tight_layout()
    plt.show()
    
    set_slm_flat(zslm._slm)

In [ ]:
# %% [markdown]
# ## 9. 关闭设备

set_slm_flat(zslm._slm)
print("✓ SLM已复位")
wfs.close()
print("✓ WFS已关闭")
zslm.close()
print("✓ SLM已关闭")
print("\n" + "="*60)
print("校准完成!")
print(f"文件: {h5_path}")
print("="*60)